Generate shell and tube geometry with CADQuery.

In [3]:
import cadquery as cq
from jupyter_cadquery import *
from jupyter_cadquery.replay import replay, enable_replay

enable_replay(show_bbox=True, warning=False)
show_object = replay

# UNITS ARE MM 

# initial cylinder parameters
R = 50
H = 500
fillet_r = 30

# initial cylinder 
cyl = cq.Workplane("front").circle(R).extrude(H)
cyl_fillet = cyl.edges("front").fillet(fillet_r).edges("back").fillet(fillet_r)

# add inlet and outlet for inner fluid
inner_fluid_r = 30
inner_fluid_depth = 50

inlet_outlet = cyl_fillet.edges("back").circle(inner_fluid_r).extrude(-inner_fluid_depth).edges("front").circle(inner_fluid_r).extrude(inner_fluid_depth)

# add inlet and outlet for outer fluid
outer_fluid_r = 15
outer_fluid_depth = 100
outer_fluid_inlet_z = H*0.8 - H/2 # 80 percent of the cylinders length
outer_fluid_outlet_z = H*0.2- H/2 # 20 percent of the cylinders length

of_inlet_wp = (
    inlet_outlet
    .workplane(offset=outer_fluid_inlet_z)             # move to middle of cylinder length
    .transformed(rotate=(90, 0, 0))    # rotate plane to face the cylinder wall
    .center(0, 0)                      # move outward to the shell surface
)

of_outlet_wp = (
    inlet_outlet
    .workplane(offset=outer_fluid_outlet_z)             # move to middle of cylinder length
    .transformed(rotate=(270, 0, 0))    # rotate plane to face the cylinder wall
    .center(0, 0)                      # move outward to the shell surface
)

of_inlet = of_inlet_wp.circle(outer_fluid_r).extrude(outer_fluid_depth)
of_outlet = of_outlet_wp.circle(outer_fluid_r).extrude(outer_fluid_depth)

inner_outer_cyl = inlet_outlet+ of_inlet + of_outlet

# tag surfaces
inner_outer_cyl.faces("+Z").tag("inner_outlet")
inner_outer_cyl.faces("-Z").tag("inner_inlet")
inner_outer_cyl.faces("+Y").tag("outer_inlet")
inner_outer_cyl.faces("-Y").tag("outer_outlet")

def generate_hole_positions(holes_per_row, col_spacing, row_spacing):
    """
    holes_per_row: list of ints, e.g. [3, 4, 3]
    col_spacing: spacing between columns
    row_spacing: spacing between rows
    """
    holes = []
    n_rows = len(holes_per_row)

    # center rows vertically around y=0
    row_offsets = [
        (i - (n_rows - 1) / 2) * row_spacing
        for i in range(n_rows)
    ]

    for row_idx, n_holes in enumerate(holes_per_row):
        y = row_offsets[row_idx]

        # center columns horizontally
        col_offsets = [
            (i - (n_holes - 1) / 2) * col_spacing
            for i in range(n_holes)
        ]

        for x in col_offsets:
            holes.append((x, y))

    return holes

# baffle parameters
baffle_radius = R
baffle_thickness = 10
baffle_start = fillet_r + H/20
offset = 0 #3*R
baffle_end = H - H/20 - fillet_r

# hole parameters
hole_diameter = 15
col_spacing = baffle_radius / 2     # auto-tied to baffle size
row_spacing = baffle_radius / 2
holes_per_row = [3, 4, 3]           # change to anything you want

# generate hole positions
holes = generate_hole_positions(
    holes_per_row,
    col_spacing=col_spacing,
    row_spacing=row_spacing
)

baffle = (
    cq.Workplane("front")
    .circle(baffle_radius)
    .extrude(baffle_thickness)
)

baffle1 = baffle.translate((offset, 0, baffle_start))
baffle2 = baffle.translate((offset, 0, baffle_end))

pipes = []

dz = baffle_end - baffle_start + baffle_thickness

# make a pipe at each hole position
for (x, y) in holes:
    pipe = (
        cq.Workplane("front")
        .workplane(offset=baffle_start)
        .center(x, y)
        .circle(hole_diameter/2)     # same size as the holes or slightly smaller
        .extrude(dz)                 # goes to the 2nd baffle
    )
    pipes.append(pipe)

# combine unshelled pipes into one object for cutting baffles
pipes_for_cut = pipes[0]  
for p in pipes[1:]:
    pipes_for_cut = pipes_for_cut.union(p)

pipes_for_cut = pipes_for_cut.translate((offset, 0, 0))
num_semi_baffles = 4   
pipe_radius = hole_diameter / 2

dz = baffle_end - baffle_start
semi_spacing = dz / (num_semi_baffles + 1)
num_semi_baffles = 4  
pipe_radius = hole_diameter / 2

dz = baffle_end - baffle_start
semi_spacing = dz / (num_semi_baffles + 1)

def make_semi_baffle(index, z_pos):
    """
    function to make semi-baffles (non-circular) for body of HX
    index = 0,1,2,... (for alternating pattern)
    z_pos = absolute z location
    """

    # start with circular baffle
    b = (
        cq.Workplane("front")
        .workplane(offset=z_pos)
        .circle(baffle_radius)
        .extrude(baffle_thickness)
    )

    # MAKE CUT SHAPE
    # annular region between inner pipe radius and outer baffle radius
    annulus = (
        cq.Workplane("front")
        .workplane(offset=z_pos)
        .circle(baffle_radius)      # outer
        # .circle(pipe_radius)        # inner
        .extrude(baffle_thickness)
    )

    # define half-plane cut direction
    cut_selector = cq.selectors.BoxSelector

    if index % 2 == 0:
        # EVEN index → remove UPPER section
        cut_box = (
            cq.Workplane("front")
            .workplane(offset=z_pos)
            .rect(2*baffle_radius, 2*baffle_radius)
            .extrude(baffle_thickness)
            .translate((0, baffle_radius/2, 0))  # upper half
        )
    else:
        # ODD index → remove LOWER section
        cut_box = (
            cq.Workplane("front")
            .workplane(offset=z_pos)
            .rect(2*baffle_radius, 2*baffle_radius)
            .extrude(baffle_thickness)
            .translate((0, -baffle_radius/2, 0))  # lower half
        )

    # intersect annulus with half-plane box to get the semi-shape
    cut_shape = annulus.intersect(cut_box)

    # subtract it from the full baffle
    semi_baffle = cut_shape   # keep only the half-annulus


    return semi_baffle
    
semi_baffles = []

for i in range(num_semi_baffles):
    z = baffle_start + (i + 1) * semi_spacing
    sb = make_semi_baffle(i, z)
    cut_sb = sb.cut(pipes_for_cut)
    semi_baffles.append(cut_sb)

semi_group = cq.Compound.makeCompound([sb.val() for sb in semi_baffles])
semi_group = semi_group.translate((offset, 0, 0))

# CUT PIPES FROM BAFFLES
baffle1 = baffle1.cut(pipes_for_cut)
baffle2 = baffle2.cut(pipes_for_cut)    

# now shell pipes 
pipes_all = pipes[0].faces(">Z or <Z").shell(-1)  
for p in pipes[1:]:
    shelled = p.faces(">Z or <Z").shell(-1)  # make them hollow
    pipes_all = pipes_all.union(shelled)

pipes_all = pipes_all.translate((offset, 0, 0))

# combined inner wall structure
wall_structure = baffle1 + baffle2 + pipes_all + semi_group

# now need to cut wall structure from inner_outer_cyl 
fluid_volumes = inner_outer_cyl.cut(wall_structure)
# TODO: figure out how to tag two volumes and export as .stl

hx = cq.Assembly() #  WHATEVER IS ADDED TO ASSEMBLY IS VISUALIZED BELOW

hx.add(wall_structure, name="wall_structure", color=cq.Color("gray"))
hx.add(fluid_volumes, name="fluid_volumes", color=cq.Color("green", alpha=0.3))
hx.save("hx.step")




Enabling jupyter_cadquery replay
+

/Users/kaelyndunnell/miniforge3/envs/fusion-hx-env/lib/python3.12/site-packages/cadquery/utils.py:40: FutureWarning: save will be removed in the next release.
  warn(f"{f.__name__} will be removed in the next release.", FutureWarning)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

ConnectionError: HTTPConnectionPool(host='localhost', port=8888): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8888): Failed to establish a new connection: [Errno 61] Connection refused"))

Tag surfaces with GMSH and export each to .stl for use in snappyHexMesh with OpenFOAM.

In [5]:
import gmsh

###############################################
###### CREATE .STL FILES FROM CAD MODEL ######
###############################################

# LOAD CAD AND INITIALIZE MESH

gmsh.initialize()
gmsh.option.setString(
    "Geometry.OCCTargetUnit", "MM"
)  # make sure gmsh reads .step file in mm, what CADQuery exports in
gmsh.model.add("openfoam_mesh")

cad_file_path = "hx.step"

entities = gmsh.model.occ.importShapes(cad_file_path)
gmsh.model.occ.synchronize()

# EXTRACT ALL VOLUMES
volumes = [e for e in gmsh.model.occ.getEntities() if e[0] == 3]

print(f"Extracted {len(volumes)} raw volumes from CAD.")

#### FRAGMENT VOLUMES & GENERATE SHARED SURFACES #####
print("Fragmenting volumes to define interfaces...")
gmsh.model.occ.fragment(volumes, [])
gmsh.model.occ.synchronize()

# FINAL VOLUMES AFTER FRAGMENT
final_volumes = gmsh.model.getEntities(dim=3)
print(f"Final number of volumes: {len(final_volumes)}")

# interface surfaces
surfaces = gmsh.model.getEntities(dim=2)

walls_coolant_interfaces = [2,4,5,6,7,8,9,10,11,12,13,24]
walls_breeder_interfaces = [3,14,15,16,17,18,19,20,21,22,23,25,26,27,28,29,30,31,33,34,35,36,37,38,39,41,42,43,44,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,71,72,73,74,75,76,77,79,80,81,82]
coolant_inlet = 85
coolant_outlet = 92
coolant_walls = [83,84,86,87,88,89,90,91]
breeder_inlet = 97
breeder_outlet = 96
breeder_walls = [93,94,95]
walls = [1,32,40,45,70,78]

coolant_inlet_marker = 6
coolant_outlet_marker = 7
coolant_walls_marker = 8
breeder_inlet_marker = 9
breeder_outlet_marker = 10
breeder_walls_marker = 11
walls_marker = 12

# .STL FILE GENERATION FOR OPENFOAM

# iterate over each interface surface and export to separate .stl file 
for surface in surfaces:
    tag = surface[1]

    if tag in walls_coolant_interfaces:
        name = "walls_coolant_interface"
    elif tag in walls_breeder_interfaces:
        name = "walls_breeder_interface"
    else:
        continue  # skip unneeded surfaces

    # add to temp physical group
    gmsh_tag = gmsh.model.addPhysicalGroup(2, [tag])

    # stl binary format 
    gmsh.option.setNumber("Mesh.Binary", 1) # 1 for binary, 0 for ASCII

    # filename for export 
    filename = f"{name}_surface.stl"

    # write mesh for each physical group
    gmsh.write(filename)
    print(f"Exported {filename}")

    # remove physical group so can restart
    gmsh.model.removePhysicalGroups([(2, gmsh_tag)])

for volume in final_volumes:
    tag = volume[1]

    if tag == 1 :
        name = "walls"
        gmsh.model.addPhysicalGroup(2, walls, walls_marker, name="walls") # add surfaces for each vol
    elif tag == 2:
        name = "coolant"
        gmsh.model.addPhysicalGroup(2, [coolant_inlet], coolant_inlet_marker, name="coolant_inlet")
        gmsh.model.addPhysicalGroup(2, [coolant_outlet], coolant_outlet_marker, name="coolant_outlet")
        gmsh.model.addPhysicalGroup(2, coolant_walls, coolant_walls_marker, name="coolant_walls")
    elif tag == 3:
        name = "breeder"
        gmsh.model.addPhysicalGroup(2, [breeder_inlet], breeder_inlet_marker, name="breeder_inlet")
        gmsh.model.addPhysicalGroup(2, [breeder_outlet], breeder_outlet_marker, name="breeder_outlet")
        gmsh.model.addPhysicalGroup(2, breeder_walls, breeder_walls_marker, name="breeder_walls")

    # add to temp physical group
    gmsh_tag = gmsh.model.addPhysicalGroup(3, [tag])

    # stl binary format 
    gmsh.option.setNumber("Mesh.Binary", 1) # 1 for binary, 0 for ASCII

    # filename for export 
    filename = f"{name}_volume.stl"

    # write mesh for each physical group
    gmsh.write(filename)
    print(f"Exported {filename}")

    # remove physical group so can restart
    gmsh.model.removePhysicalGroups([(3, gmsh_tag)])

    surface_physical_groups = gmsh.model.getPhysicalGroups(2)
    for group in surface_physical_groups:
        rm_tag = group[1] 
        gmsh.model.removePhysicalGroups([(2, rm_tag)])

##### TAG & NAME PHYSICAL GROUPS #####
# need to open mesh in gmsh gui to determine the proper tagging as below

# volumes
walls_marker = 1
coolant_marker = 2
breeder_marker = 3

gmsh.model.addPhysicalGroup(3, [1], walls_marker, name=f"walls")
gmsh.model.addPhysicalGroup(3, [2], coolant_marker, name=f"coolant")
gmsh.model.addPhysicalGroup(3, [3], breeder_marker, name=f"breeder")

# surfaces between walls and coolant 
walls_coolant_interface_marker = 4
gmsh.model.addPhysicalGroup(2, walls_coolant_interfaces, walls_coolant_interface_marker, name="walls_coolant_interface")

# surfaces between walls and breeder
walls_breeder_interface_marker = 5
gmsh.model.addPhysicalGroup(2, walls_breeder_interfaces, walls_breeder_interface_marker, name="walls_breeder_interface")


# other surfaces

coolant_inlet_marker = 6
coolant_outlet_marker = 7
breeder_inlet_marker = 8
breeder_outlet_marker = 9
walls_marker = 10

gmsh.model.addPhysicalGroup(2, walls, walls_marker, name="walls")
gmsh.model.addPhysicalGroup(2, [coolant_inlet], coolant_inlet_marker, name="coolant_inlet")
gmsh.model.addPhysicalGroup(2, [coolant_outlet], coolant_outlet_marker, name="coolant_outlet")
gmsh.model.addPhysicalGroup(2, [breeder_inlet], breeder_inlet_marker, name="breeder_inlet")
gmsh.model.addPhysicalGroup(2, [breeder_outlet], breeder_outlet_marker, name="breeder_outlet")


# ##### MESH SIZE & REFINEMENT #####
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 10)

##### SYNC & GENERATE MESH #####
gmsh.model.occ.synchronize()
gmsh.model.mesh.generate(3)

##### SAVE MESH #####
output_file = "hx.msh"
gmsh.write(output_file)
gmsh.finalize()


Info    :  - Label 'Shapes/a19f6406-f21a-11f0-a015-ba5fe48434ce/wall_structure/wall_structure' (3D)
Info    :  - Color (0.752941, 0.752941, 0.752941) (3D & Surfaces)
Info    :  - Label 'Shapes/a19f6406-f21a-11f0-a015-ba5fe48434ce/fluid_volumes/fluid_volumes' (3D)
Info    :  - Label 'Shapes/fluid_volumes' (3D)
Info    :  - Color (0, 1, 0) (3D & Surfaces)
Info    :  - Label 'Shapes/fluid_volumes' (3D)
Info    :  - Color (0, 1, 0) (3D & Surfaces)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_volumes' (2D)
Info    :  - Label 'Shapes/fluid_vo